# Import/Settings

In [3]:
# Заставляем ноутбук обновлять импорты автоматически (если ты изменил код в src/)
%load_ext autoreload
%autoreload 2

In [4]:
# 1. Стандартная библиотека
import sys
import warnings
from pathlib import Path

# 2. Сторонние библиотеки
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from hydra import initialize, compose
import sweetviz as sv

# 3. Локальные модули
sys.path.append(str(Path.cwd().parent))
from src.core.data import UniversalDataLoader

# --- РАБОТА С ПРЕДУПРЕЖДЕНИЯМИ ---
warnings.filterwarnings('ignore', category=FutureWarning)

# --- ИНИЦИАЛИЗАЦИЯ HYDRA ---
with initialize(version_base=None, config_path="../configs"):
    cfg = compose(config_name="config")

print(f"Проект: {cfg.project_name} | Режим: EDA")

# --- НАСТРОЙКИ ВИЗУАЛИЗАЦИИ ---
try:
    plots_cfg = cfg.logging.plots

    plt.style.use(plots_cfg.style)
    plt.rcParams['figure.figsize'] = plots_cfg.fig_size
    plt.rcParams['figure.dpi'] = plots_cfg.dpi
    plt.rcParams['font.size'] = plots_cfg.font_size
    plt.rcParams['axes.grid'] = plots_cfg.grid
    plt.rcParams['axes.spines.top'] = plots_cfg.spines_top
    plt.rcParams['axes.spines.right'] = plots_cfg.spines_right

    # Сохраним alpha как константу для использования прямо в функциях отрисовки,
    # так как matplotlib не умеет задавать прозрачность глобально
    PLOT_ALPHA = plots_cfg.alpha

    print("Глобальные настройки графиков успешно применены из конфига!")
except AttributeError:
    print("Внимание: Блок logging.plots не найден в конфиге. Используются стандартные графики.")

Проект: outsource_project_name | Режим: EDA
Глобальные настройки графиков успешно применены из конфига!


# DataLoading

In [ ]:
# Вся грязная работа (джоины, сэмплирование) скрыта внутри лоадера
loader = UniversalDataLoader(cfg)
df = loader.load_data()

print(f"Размер датасета: {df.shape}")
print("-" * 30)
# Быстрый чек глазами: не отвалились ли фичи, правильные ли типы
df.info()

# EDA

## Base Check-Up

In [ ]:
# Смотрим, где больше всего дыр в данных
missing = df.isnull().mean() * 100
missing = missing[missing > 0].sort_values(ascending=False)

if not missing.empty:
    print("Топ колонок с пропусками (%):")
    print(missing.head(10))
else:
    print("Пропусков нет! Идеально.")

df.info()

## Targer-Analise

In [ ]:
target_col = cfg.data.tabular.target_col
task_type = cfg.data.tabular.task_type

plt.figure(figsize=(8, 4))

if task_type in ['binary', 'multiclass']:
    # Смотрим дисбаланс классов
    sns.countplot(data=df, x=target_col)
    plt.title("Распределение классов (Таргет)")
elif task_type == 'regression':
    # Смотрим распределение (хвосты, выбросы)
    sns.histplot(data=df, x=target_col, kde=True, bins=50)
    plt.title("Распределение целевой переменной (Регрессия)")

plt.show()

## AutoEDA

In [ ]:
print("Генерация HTML-отчета через Sweetviz...")

# Запускаем тяжелую артиллерию
report = sv.analyze(df, target_feat=cfg.data.tabular.target_col)

reports_dir = Path(cfg.paths.reports_dir)
reports_dir.mkdir(parents=True, exist_ok=True)

output_path = reports_dir / "eda_report.html"
report.show_html(filepath=str(output_path), open_browser=True)

print(f"Отчет сохранен в: {output_path}")

## Custom analise

In [8]:
# ==========================================
# ЗДЕСЬ МЕСТО ДЛЯ КАСТОМНОЙ БИЗНЕС-ЛОГИКИ
# (Сложные группировки, нестандартные гипотезы заказчика,
# которые не покрывает автоматический отчет Sweetviz)
# ==========================================
